In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import skimage

from skimage import io
import numpy as np
import io
from PIL import Image, ImageFilter
import cv2
from skimage import measure
import os

def dekodiranje(image_path):

    img = cv2.imread(image_path)

    image = np.copy(img)
    white_tresh = np.array([100, 100, 100])
    colored_pixels = np.any(image < white_tresh, axis=-1)
    image[colored_pixels] = [0, 0, 0]

    blur_img = cv2.GaussianBlur(image, (25, 25), 0)


    blur_img_rgb = cv2.cvtColor(blur_img, cv2.COLOR_BGR2RGB)
    gray_image = cv2.cvtColor(blur_img_rgb, cv2.COLOR_BGR2GRAY)
    kernel = np.ones((4, 4), np.uint8)
    img_closed = np.copy(gray_image)
    img_closed = cv2.dilate(img_closed, kernel, iterations=4)
    img_closed = cv2.erode(img_closed, kernel, iterations=6)


    """ret, thresh = cv2.threshold(img_closed, 120, 255, 0) #cv2.THRESH_BINARY um 0
    plt.imshow(thresh)
    plt.show()
    #contours, hierarchy = cv2.findContours(image=thresh, mode=cv2.RETR_TREE, method=cv2.CHAIN_APPROX_NONE)


    #cv2.drawContours(image=image_copy, contours=contours, contourIdx=-1, color=(0, 255, 0), thickness=5, lineType=cv2.LINE_AA)
    plt.imshow(image_copy)
    plt.show()"""

    ret, thresh = cv2.threshold(img_closed, 110, 255, cv2.THRESH_BINARY)
    cont = np.copy(img)


    contours, hierarchy = cv2.findContours(image=thresh, mode=cv2.RETR_TREE, method=cv2.CHAIN_APPROX_NONE)



    def expected_perimeter(w, h):
        return 2 * (w + h)

    max_gapprc = 30

    for cnt in contours:
        approx = cv2.approxPolyDP(cnt, 0.03*cv2.arcLength(cnt, True), True)
        if len(approx) == 4:
            x, y, w, h = cv2.boundingRect(cnt)
            ratio = float(w) / h
            expected_p = expected_perimeter(w, h)
            max_gap = expected_p * (max_gapprc / 100.0)
            actual_p = cv2.arcLength(cnt, True)

            if actual_p <= expected_p + max_gap:
                cont = cv2.drawContours(image, [cnt], -1, (0, 255, 0), 9)



    contours = sorted(contours, key=cv2.contourArea, reverse=True)



    bigcont = np.copy(img)
    if contours:
        rect = [cnt for cnt in contours if len(cv2.approxPolyDP(cnt, 0.03 * cv2.arcLength(cnt, True), True)) == 4]

        if rect:
            largest_contour = max(rect, key=cv2.contourArea)
            cv2.drawContours(image=bigcont, contours=[largest_contour], contourIdx=-1, color=(255, 0, 0), thickness=7, lineType=cv2.LINE_AA)


            rect = cv2.minAreaRect(largest_contour)
            angle = rect[2]


            #if angle < -45:
            #   angle += 90



            center = rect[0]
            rotation_matrix = cv2.getRotationMatrix2D(center, angle, scale=1.0)

            rotated_image = cv2.warpAffine(bigcont, rotation_matrix, dsize=(bigcont.shape[1], bigcont.shape[0]))


    if contours:

        #rect = [cnt for cnt in contours if len(cv2.approxPolyDP(cnt, 0.1 * cv2.arcLength(cnt, True), True)) == 4]

        if rect:
            #largest_contour = max(rect, key=cv2.contourArea)

            image_copy = np.copy(image)
            cv2.drawContours(image=image_copy, contours=[largest_contour], contourIdx=-1, color=(255, 0, 0), thickness=7, lineType=cv2.LINE_AA)

            rect = cv2.minAreaRect(largest_contour)
            box = cv2.boxPoints(rect)
            box = np.int0(box)

            angle = rect[-1]

            if angle < -45:
                angle += 90

            height, width = image_copy.shape[:2]
            center = (width // 2, height // 2)
            rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1)
            image_copy_rotated = cv2.warpAffine(img, rotation_matrix, (width, height), flags=cv2.INTER_LINEAR)


    if "image_copy_rotated" not in locals():
        encoded_string = "00000000"
    else:
        white_color_threshold = np.array([100, 100, 100])
        colored_pixels = np.any(image_copy_rotated < white_color_threshold, axis=-1)
        image_copy_rotated[colored_pixels] = [0, 0, 0]
        blur_img = cv2.GaussianBlur(image_copy_rotated, (25, 25), 0)

        blur_img_rgb = cv2.cvtColor(blur_img, cv2.COLOR_BGR2RGB)
        gray_image = cv2.cvtColor(blur_img_rgb, cv2.COLOR_BGR2GRAY)
        kernel = np.ones((3, 3), np.uint8)
        img_closed = np.copy(gray_image)
        img_closed = cv2.dilate(img_closed, kernel, iterations=4)
        img_closed = cv2.erode(img_closed, kernel, iterations=6)




        ret,thresh = cv2.threshold(img_closed ,120,255, cv2.THRESH_BINARY)
        contours, hierarchy = cv2.findContours(image=thresh, mode=cv2.RETR_TREE, method=cv2.CHAIN_APPROX_NONE)




        #jer dole
        img_copy90 = np.copy(image_copy_rotated)
        img_copy99=np.copy(image_copy_rotated)
        img_copy109=np.copy(image_copy_rotated)

        for cnt in contours:
            x1,y1 = cnt[0][0]
            approx = cv2.approxPolyDP(cnt, 0.01*cv2.arcLength(cnt, True), True)
            if len(approx) == 4:
                x, y, w, h = cv2.boundingRect(cnt)
                ratio = float(w)/h
                if ratio >= 0.9 and ratio <= 1.1:
                    image_copy_rotated = cv2.drawContours(image_copy_rotated, [cnt], -1, (0,255,255), 9)
                    cv2.putText(image_copy_rotated, 'Square', (x1, y1), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                else:
                    cv2.putText(image_copy_rotated, 'Rectangle', (x1, y1), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                    image_copy_rotated = cv2.drawContours(image_copy_rotated, [cnt], -1, (0,255,0), 9)



        if contours:

            rect = [cnt for cnt in contours if len(cv2.approxPolyDP(cnt, 0.01 * cv2.arcLength(cnt, True), True)) == 4]

            if rect:

                largest_contour = max(rect, key=cv2.contourArea)

                x, y, w, h = cv2.boundingRect(largest_contour)

                if h > w:
                    cropped_image = img_copy109[y:y+h, x:x+w]
                    cropped_image = cv2.rotate(cropped_image, cv2.ROTATE_90_CLOCKWISE)
                else:
                    cropped_image = img_copy109[y:y+h, x:x+w]
        if "cropped_image" not in locals():
            encoded_string="0000000"
        else:

            threshold_value = 120
            _, binary_image = cv2.threshold(cropped_image, threshold_value, 130, cv2.THRESH_BINARY)
            gray_image = cv2.cvtColor(binary_image, cv2.COLOR_BGR2GRAY)
            contours, hierarchy = cv2.findContours(image=gray_image, mode=cv2.RETR_TREE, method=cv2.CHAIN_APPROX_NONE)


            binary_image_copy = binary_image.copy()

            height = binary_image.shape[0]
            detected_contours = []

            for contour in contours:
                area = cv2.contourArea(contour)
                x, y, w, h = cv2.boundingRect(contour)
                cx, cy = x + w // 2, y + h // 2
                min_y = y + h - 1
                max_y = y

                if (min_y > height // 2 and max_y > height // 2) or (min_y < height // 2 and max_y < height // 2):
                    detected_contours.append(contour)

            cv2.drawContours(binary_image_copy, detected_contours, -1, (0), thickness=cv2.FILLED)

            for contour in detected_contours:
                x, y, w, h = cv2.boundingRect(contour)
                cv2.rectangle(cropped_image, (x, y), (x + w, y + h), (255, 255, 255), 2)
                cropped_image[y:y+h, x:x+w] = 255



            threshold_value = 120
            _, binary_image1 = cv2.threshold(cropped_image, threshold_value, 130, cv2.THRESH_BINARY)



            min_blue = np.array([50, 50, 50])
            max_blue = np.array([150, 255, 255])

            hsv_image = cv2.cvtColor(binary_image1, cv2.COLOR_RGB2HSV)

            mask = cv2.inRange(hsv_image, min_blue, max_blue)

            binary_image1[mask > 0] = (255, 255, 255)


            threshold_value = 120
            _, bar_code = cv2.threshold(binary_image1, threshold_value, 150, cv2.THRESH_BINARY)

            def calculate_average_y(highest_points):
                return np.mean([point[1] for point in highest_points])

            gray_image = cv2.cvtColor(bar_code, cv2.COLOR_BGR2GRAY)
            kernel = np.ones((2, 2), np.uint8)
            img_closed = np.copy(gray_image)
            img_closed = cv2.dilate(img_closed, kernel, iterations=2)
            img_closed = cv2.erode(img_closed, kernel, iterations=4)




            def calculate_average_y(highest_points):
                return np.mean([point[1] for point in highest_points])



            threshold_value = 120
            _, binary_image = cv2.threshold(bar_code, threshold_value, 255, cv2.THRESH_BINARY)
            gray_image = cv2.cvtColor(binary_image, cv2.COLOR_BGR2GRAY)
            contours, hierarchy = cv2.findContours(image=gray_image, mode=cv2.RETR_TREE, method=cv2.CHAIN_APPROX_NONE)

            if len(contours) == 0 :
                encoded_string= "1011010100101000100"
            else:

                largest_contour_index = np.argmax([cv2.contourArea(cnt) for cnt in contours])

                barcode = bar_code.copy()

                highest_y_coordinate = contours[largest_contour_index][contours[largest_contour_index][:, :, 1].argmin()][0][1]

                all_highest_points = []
                all_y_coordinates = []
                for i, cnt in enumerate(contours):
                    if i != largest_contour_index:
                        topmost_point = tuple(cnt[cnt[:, :, 1].argmin()][0])
                        all_highest_points.append(topmost_point)
                        all_y_coordinates.append(topmost_point[1])

                if len(all_y_coordinates) < 1:
                    encoded_string='1010010000100'
                else:
                    most_common_y = np.bincount(all_y_coordinates).argmax()

                    red_dots_count = 0

                    for i, point in enumerate(all_highest_points):
                        if abs(point[1] - most_common_y) < 20:
                            cv2.circle(barcode, point, 5, (0, 0, 255), -1)
                        elif point[1] < most_common_y:
                            cv2.circle(barcode, point, 5, (255, 0, 0), -1)
                            red_dots_count += 1



                    if red_dots_count == 6:
                        barcode = cv2.rotate(bar_code, cv2.ROTATE_180)

                    else:
                        bar_code = cv2.rotate(bar_code, cv2.ROTATE_180)
                        barcode = cv2.rotate(bar_code, cv2.ROTATE_180)

                    gray = cv2.cvtColor(barcode,cv2.COLOR_BGR2GRAY)

                    kernel_size = 5
                    #blur_gray = cv2.GaussianBlur(gray,(kernel_size, kernel_size),0)

                    low_threshold = 0
                    high_threshold = 200
                    edges = cv2.Canny(gray, low_threshold, high_threshold)

                    rho = 1
                    theta = np.pi / 180
                    threshold = 15
                    min_line_length = 100
                    max_line_gap = 160
                    line_image = np.copy(barcode) * 0


                    lines = cv2.HoughLinesP(edges, rho, theta, threshold, np.array([]), min_line_length, max_line_gap)
                    if lines is None:
                        encoded_string='00000000000'
                    else:

                        for line in lines:
                            for x1,y1,x2,y2 in line:
                                cv2.line(line_image,(x1,y1),(x2,y2),(255,0,0),2)
                        lines_edges = cv2.addWeighted(barcode, 1, line_image, 1, 0)


                        angles = []
                        for line in lines:
                            for x1, y1, x2, y2 in line:
                                angle = np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi
                                angles.append(angle)

                        # sa slicnim uglom, paralelne
                        angle_threshold = 5
                        groups = []
                        for angle in angles:
                            found_group = False
                            for group in groups:
                                if abs(angle - group[0]) < angle_threshold:
                                    group.append(angle)
                                    found_group = True
                                    break
                            if not found_group:
                                groups.append([angle])

                        max_group = max(groups, key=len)

                        filtered_lines = [line for line, angle in zip(lines, angles) if angle in max_group]

                        line_image_filtered = np.copy(barcode) * 0

                        for line in filtered_lines:
                            for x1, y1, x2, y2 in line:
                                cv2.line(line_image_filtered, (x1, y1), (x2, y2), (0, 0, 255), 2)

                        lines_edges_filtered = cv2.addWeighted(barcode, 1, line_image_filtered, 1, 0)


                        gray = cv2.cvtColor(barcode, cv2.COLOR_BGR2GRAY)

                        low_threshold = 0
                        high_threshold = 200
                        edges = cv2.Canny(gray, low_threshold, high_threshold)

                        rho = 1
                        theta = np.pi / 180
                        threshold = 15
                        min_line_length = 100
                        max_line_gap = 160
                        lines = cv2.HoughLinesP(edges, rho, theta, threshold, np.array([]), min_line_length, max_line_gap)
                        if lines is None:
                            encoded_string='000000'
                        else:

                            angles = []
                            for line in lines:
                                for x1, y1, x2, y2 in line:
                                    angle = np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi
                                    angles.append(angle)

                            angle_threshold = 5
                            groups = []
                            for angle in angles:
                                found_group = False
                                for group in groups:
                                    if abs(angle - group[0]) < angle_threshold:
                                        group.append(angle)
                                        found_group = True
                                        break
                                if not found_group:
                                    groups.append([angle])

                            max_group = max(groups, key=len)

                            avg_angle = np.mean(max_group)


                            rows, cols = barcode.shape[:2]
                            rotation_matrix = cv2.getRotationMatrix2D((cols / 2, rows / 2), avg_angle + 90, 1)
                            rotated_image = cv2.warpAffine(barcode, rotation_matrix, (cols, rows), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=255
                                                        )


                            line_image_filtered = np.copy(rotated_image) * 0

                            for line in lines:
                                for x1, y1, x2, y2 in line:
                                    angle = np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi
                                    if abs(angle - avg_angle) < angle_threshold:
                                        cv2.line(line_image_filtered, (x1, y1), (x2, y2), (0, 0, 255), 2)


                            lines_edges_filtered = cv2.addWeighted(rotated_image, 1, line_image_filtered, 1, 0)



                            threshold_value = 120
                            _, binary_image = cv2.threshold(rotated_image, threshold_value, 150, cv2.THRESH_BINARY)
                            gray_image = cv2.cvtColor(binary_image, cv2.COLOR_BGR2GRAY)
                            contours, hierarchy = cv2.findContours(image=gray_image, mode=cv2.RETR_TREE, method=cv2.CHAIN_APPROX_NONE)

                            rotated_image_copy = rotated_image.copy()

                            valid_contours = []
                            for contour in contours:
                                area = cv2.contourArea(contour)
                                if 800 < area < 8000:
                                    cv2.drawContours(rotated_image, [contour], -1, (255), thickness=cv2.FILLED)
                                    valid_contours.append(contour)


                            if valid_contours:
                                x, y, w, h = cv2.boundingRect(np.concatenate(valid_contours))
                                cv2.rectangle(rotated_image_copy, (x, y), (x + w, y + h), (255), 2)

                                rotated_image = binary_image[y:y+h, x:x+w]



                            if len(rotated_image.shape) == 3:
                                bar_code = cv2.cvtColor(rotated_image, cv2.COLOR_BGR2GRAY)

                            height, width = bar_code.shape

                            middle_y = height // 2

                            threshold_value = 128

                            def find_distance_between_first_and_last_blacks():
                                first_black_x = None
                                last_black_x = None
                                for x in range(width):
                                    pixel_value = bar_code[middle_y, x]
                                    is_black = pixel_value < threshold_value

                                    if is_black:
                                        if first_black_x is None:
                                            first_black_x = x
                                        last_black_x = x

                                if first_black_x is not None and last_black_x is not None:
                                    return last_black_x - first_black_x

                                return None

                            distance_between_blacks = find_distance_between_first_and_last_blacks()
                            #sa 95 jer toliko ima fvuniwurbvuib
                            unit_of_measure = distance_between_blacks // 105

                            binary_values = []

                            if unit_of_measure == 0:
                                binary_string = '000000000000000000000000000000000000000000000000000000000000000000000000000000000000'
                            else:
                                def append_binary_value(distance, is_black):
                                    max_consecutive = 4
                                    num_units = min(distance // unit_of_measure, max_consecutive)

                                    # ako je zbog lose slikke koda neka jedinicna praznina manja od jedinice mere
                                    if num_units == 0:
                                        if is_black:
                                            binary_values.append('1')
                                        else:
                                            binary_values.append('0')
                                    else:
                                        binary_values.append('1' * num_units if is_black else '0' * num_units)

                                current_color = None
                                consecutive_count = 0
                                start_from_black = False
                                for x in range(width):
                                    pixel_value = bar_code[middle_y, x]
                                    is_black = pixel_value < threshold_value

                                    if current_color is None and not is_black:
                                        continue

                                    if current_color is None:
                                        start_from_black = True
                                        current_color = is_black
                                        consecutive_count = 1
                                    elif current_color == is_black:
                                        consecutive_count += 1
                                    else:
                                        if start_from_black:
                                            append_binary_value(consecutive_count, current_color)
                                        current_color = is_black
                                        consecutive_count = 1

                                if current_color == True:
                                    append_binary_value(consecutive_count, current_color)

                                binary_string = ''.join(binary_values)

                            if binary_string.startswith('101'):
                                binary_string = binary_string[3:]
                            if binary_string.endswith('101'):
                                binary_string = binary_string[:-3]

                            #ybog centra koji se ne gleda
                            binary_string = binary_string.replace('1010101', '11')
                            #ako dlucajno detektuje vise da povecam tacnost
                            binary_string = binary_string.replace('11111', '1111')
                            binary_string = binary_string.replace('111111', '1111')
                            binary_string = binary_string.replace('00000', '0000')
                            binary_string = binary_string.replace('000000', '0000')




                            def encode_digit(digit_binary_str):
                                scheme_a = {
                                    '0001101': '0',
                                    '0011001': '1',
                                    '0010011': '2',
                                    '0111101': '3',
                                    '0100011': '4',
                                    '0110001': '5',
                                    '0101111': '6',
                                    '0111011': '7',
                                    '0110111': '8',
                                    '0001011': '9',
                                }

                                scheme_b = {
                                    '0100111': '0',
                                    '0110011': '1',
                                    '0011011': '2',
                                    '0100001': '3',
                                    '0011101': '4',
                                    '0111001': '5',
                                    '0000101': '6',
                                    '0010001': '7',
                                    '0001001': '8',
                                    '0010111': '9',
                                }

                                scheme_c = {
                                    '1110010': '0',
                                    '1100110': '1',
                                    '1101100': '2',
                                    '1000010': '3',
                                    '1011100':'4',
                                    '1001110' : '5',
                                    '1010000': '6',
                                    '1000100': '7',
                                    '1001000':  '8',
                                    '1110100': '9',


                                }


                                if digit_binary_str in scheme_b:
                                    return scheme_b[digit_binary_str], 'b'
                                elif digit_binary_str in scheme_a:
                                    return scheme_a[digit_binary_str], 'a'
                                elif digit_binary_str in scheme_c:
                                    return scheme_c[digit_binary_str], 'c'
                                else:
                                    return 'X', 'X'


                            binary_segments = binary_string.split(' ')

                            encoded_string = ''
                            schemes_used = ''

                            for i, segment in enumerate(binary_segments):
                                if i > 0:
                                    encoded_string += ' '
                                    schemes_used += ' '

                                segment_parts = [segment[j:j+7] for j in range(0, len(segment), 7)]
                                for j, part in enumerate(segment_parts):
                                    encoded_digit, scheme_used = encode_digit(part)
                                    encoded_string += encoded_digit
                                    schemes_used += scheme_used

                            #za prvu cifru
                            if schemes_used[:6] == 'aaaaaa':
                                encoded_string = '0' + encoded_string
                            else:
                                schemes_to_numbers = {
                                    'aababb': '1',
                                    'aabbab': '2',
                                    'aabbba': '3',
                                    'abaabb': '4',
                                    'abbaab': '5',
                                    'abbbaa': '6',
                                    'ababab': '7',
                                    'ababba': '8',
                                    'abbaba': '9'
                                }
                                first_six_schemes = schemes_used[:6]
                                if first_six_schemes in schemes_to_numbers:
                                    number = schemes_to_numbers[first_six_schemes]
                                    encoded_string = number + encoded_string

                            if len(encoded_string) < 13:
                                br=13-len(encoded_string)
                                encoded_string=encoded_string+"X"*br




                            return encoded_string




def izr_tacnost(decoded, correct):


    correct_digits = 0
    total_digits = len(decoded)

    for d_decoded, d_correct in zip(decoded, correct):
        if d_decoded == d_correct:
            correct_digits += 1

    return (correct_digits / total_digits) * 100

image_folder = "Dataset1"
total_images = 0
total_digit_accuracy = 0

for filename in os.listdir(image_folder):
    if filename.endswith(".jpg") or filename.endswith(".png"):
        total_images += 1

        image_path = os.path.join(image_folder, filename)
        notepad_path = os.path.join(image_folder, filename.split('.')[0] + ".jpg.txt")

        with open(notepad_path, 'r') as f:
            correct_result = f.read().strip()

        decoded_result = dekodiranje(image_path)

        if decoded_result is None:
            image_digit_accuracy = 0
            total_digit_accuracy += image_digit_accuracy
        else:
            image_digit_accuracy = izr_tacnost(decoded_result, correct_result)
            total_digit_accuracy += image_digit_accuracy

            print(f"Slika: {filename}, Procenat Tacnosti: {image_digit_accuracy:.2f}%", {decoded_result} , {correct_result})

global_accuracy = total_digit_accuracy / total_images if total_images > 0 else 0
print("Ukupna tacnost:", global_accuracy, "%")



FileNotFoundError: ignored